# Nonlinear multi-field coupling analysis of piezoelectric semiconductors via PINNs

**Paper:** Xiao, Z., Weng, Y., Yao, W., Chen, W., Zhang, C. (2026). *Nonlinear multi-field coupling analysis of piezoelectric semiconductors via PINNs.* Science China Physics, Mechanics & Astronomy, 69(1), 214611.

**Carpeta origen:** `PINNs/1. mecanica de fluidos/Nonlinear multi-field coupling analysis of piezoelectric semiconductors via PINNs.pdf`

## Como se usan las PINNs en este paper

El paper propone **DD-PINNs-TD** (PINNs guiadas por datos con descomposicion de tareas) para modelar el acoplamiento no lineal **termo-deformacion-polarizacion-portadores (TDPC)** en semiconductores piezoelectricos (PS). En vez de una unica red que prediga todos los campos fisicos simultaneamente, el problema se **descompone por campo fisico** (Fig. 1(b) del paper): una subred independiente por cada campo (elastico $u$, electromecanico $\varphi$, portadores $n$, termico $\theta$, ...). Cada subred $k$ tiene su propia funcion de perdida (Eq. 1-3):

$$Loss_k = L_u^k + L_f^k,\qquad L_u^k=\frac{1}{N_k}\sum_i\big|u_i^{Pre}(x_i)-u_i^{Ref}(x_i)\big|^2,\qquad L_f^k=\frac{1}{N_k}\sum_j\big|f_j^k(x_j)\big|^2$$

donde $f^k$ es el residuo de la ecuacion gobernante de ese subtask (leyes constitutivas + equilibrio + Gauss + continuidad de portadores + calor, Eq. 6-9), y la perdida total es el promedio de las M subtareas (Eq. 4): $Loss=\frac{1}{M}\sum_k Loss_k$. Cuando la descomposicion es espacial (estructuras multinivel/multiescala), se an~ade una condicion de continuidad entre subtareas vecinas (Eq. 14).

Las ecuaciones constitutivas acopladas (Eq. 6-9) para un semiconductor piezoelectrico tipo n son:

$$T_{ij,j}=0,\quad D_{i,i}=q(-n+N_D^+),\quad J_{i,i}=0,\quad h_{i,i}=J_i\cdot E_i$$
$$T_{ij}=c_{ijkl}S_{kl}-e_{kij}E_k-\vartheta_{ij}\theta,\quad D_i=e_{ikl}S_{kl}+\varepsilon_{ik}E_k+p_i\theta,\quad J_i=qn\mu_iE_i+qD_i n_{,i},\quad h_i=-\kappa_i\theta_{,i}$$

es decir: equilibrio mecanico con acoplamiento piezoelectrico y termico, ley de Gauss con carga de portadores, continuidad de corriente de electrones (deriva + difusion), y conduccion de calor con calentamiento Joule.

## Simplificacion declarada

El paper trabaja con tensores 3D completos, teoria de vigas/placas con **no linealidad geometrica** (deformaciones grandes tipo von Karman) y constantes de material anisotropas del ZnO (Seccion 4). Reproducir el sistema tensorial completo (Eq. 15-37, decenas de ecuaciones acopladas) esta fuera de alcance de este cuaderno. En su lugar, implementamos un **analogo 1D lineal fiel al mecanismo DD-PINNs-TD**: una varilla piezoelectrica bajo extension axial pura (sin flexion ni no linealidad geometrica), con los mismos 4 campos acoplados ($u,\varphi,n,\theta$) y **una subred independiente por campo**, cada una con su propia perdida fisica basada en la version 1D-lineal de las Eq. 6-9. Se conserva exactamente:
- La descomposicion de tareas por campo fisico (una red por campo, Fig. 1(b) y Fig. 2).
- El acoplamiento entre subredes a traves de los residuos fisicos cruzados (cada red usa las salidas/derivadas de las otras).
- La perdida total como promedio de las perdidas de subtarea (Eq. 4).

## Repositorio publico de referencia

El PDF no incluye un repositorio de codigo propio. Como referencia publica del principio general de PINNs con descomposicion de tareas/dominio (el mismo mecanismo estructural que DD-PINNs-TD), se usa:

- **AmeyaJagtap/XPINNs** &mdash; https://github.com/AmeyaJagtap/XPINNs — PINNs con descomposicion de dominio/tarea en subredes independientes.

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Sistema TDPC 1D-lineal (analogo axial de la varilla piezoelectrica tipo ZnO, Seccion 4)

$x\in[0,a]$, extremo $x=0$ empotrado (condicion de contorno 'clamped' del paper: $u=0,\varphi=0,n=N_0,\theta=0$), carga axial $P$ en $x=a$. Version 1D-lineal de Eq. (6)-(9):

$$c\,u''+ e\,\varphi'' -\vartheta\,\theta' = 0 \quad\text{(equilibrio mecanico + piezoelectrico + termico)}$$
$$e\,u'' - \varepsilon\,\varphi'' = q\,(n-N_0) \quad\text{(ley de Gauss, con portadores respecto a la concentracion inicial } N_0\text{)}$$
$$-q\mu\,(n\varphi')' + qD_n\,n'' = 0 \quad\text{(continuidad de corriente de electrones, deriva + difusion)}$$
$$-\kappa\,\theta'' = -q\mu\,n\,(\varphi')^2 \quad\text{(calor, con calentamiento Joule }J\cdot E)$$

In [ ]:
# Parametros representativos (ordenes de magnitud tipicos de ZnO, normalizados para estabilidad numerica)
a = 1.0        # longitud normalizada de la varilla
c = 1.0        # constante elastica (normalizada)
e_piezo = 0.3  # constante piezoelectrica (normalizada)
eps = 1.0      # constante dielectrica (normalizada)
theta_stress = 0.05  # constante de esfuerzo termico
q = 1.0        # carga elemental (normalizada)
mu = 0.5       # movilidad electronica (normalizada)
Dn = 0.2       # coeficiente de difusion electronica (normalizada)
kappa = 1.0    # conductividad termica (normalizada)
N0 = 1.0       # concentracion inicial de electrones (normalizada)
P_load = 0.5   # carga axial mecanica aplicada en x=a

## 2. DD-PINNs-TD: una subred por campo fisico (Fig. 1(b), Fig. 2)

In [ ]:
class FieldNet(nn.Module):
    """Subred para un unico campo fisico (subtask FLS-k), arquitectura pequena tipo Fig. 2."""
    def __init__(self, n_hidden=3, n_neurons=32):
        super().__init__()
        layers = [nn.Linear(1, n_neurons), nn.Tanh()]
        for _ in range(n_hidden - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers += [nn.Linear(n_neurons, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


net_u = FieldNet().to(device)      # FLS-1: campo elastico (desplazamiento axial)
net_phi = FieldNet().to(device)    # FLS-2: campo electromecanico (potencial)
net_n = FieldNet().to(device)      # FLS-3: campo de portadores (concentracion de electrones)
net_theta = FieldNet().to(device)  # FLS-4: campo termico (temperatura)


def d_dx(f, x):
    return torch.autograd.grad(f, x, grad_outputs=torch.ones_like(f),
                                create_graph=True, retain_graph=True)[0]

## 3. Perdida de cada subtarea (Eq. 1-3): residuo fisico acoplado + condicion de contorno

In [ ]:
N_col = 100
x_col = torch.linspace(1e-3, a - 1e-3, N_col, device=device).view(-1, 1).requires_grad_(True)
x0 = torch.zeros(1, 1, device=device, requires_grad=True)
x_end = torch.full((1, 1), a, device=device, requires_grad=True)


def field_values(x):
    u = net_u(x)
    phi = net_phi(x)
    n = net_n(x)
    theta = net_theta(x)
    return u, phi, n, theta


def compute_losses():
    u, phi, n, theta = field_values(x_col)
    u_x = d_dx(u, x_col); u_xx = d_dx(u_x, x_col)
    phi_x = d_dx(phi, x_col); phi_xx = d_dx(phi_x, x_col)
    n_x = d_dx(n, x_col); n_xx = d_dx(n_x, x_col)
    theta_x = d_dx(theta, x_col); theta_xx = d_dx(theta_x, x_col)

    # FLS-1: equilibrio mecanico piezotermico
    res_u = c * u_xx + e_piezo * phi_xx - theta_stress * theta_x
    loss_u_phy = torch.mean(res_u**2)

    # FLS-2: ley de Gauss con portadores
    res_phi = e_piezo * u_xx - eps * phi_xx - q * (n - N0)
    loss_phi_phy = torch.mean(res_phi**2)

    # FLS-3: continuidad de corriente de electrones (deriva + difusion)
    drift = n * phi_x
    drift_x = d_dx(drift, x_col)
    res_n = -q * mu * drift_x + q * Dn * n_xx
    loss_n_phy = torch.mean(res_n**2)

    # FLS-4: conduccion de calor con calentamiento Joule J.E
    res_theta = -kappa * theta_xx - q * mu * n * phi_x**2
    loss_theta_phy = torch.mean(res_theta**2)

    # Condiciones de contorno (empotrado en x=0, carga mecanica en x=a)
    u0, phi0, n0, theta0 = field_values(x0)
    loss_bc0 = torch.mean(u0**2 + phi0**2 + (n0 - N0)**2 + theta0**2)

    u_end, _, _, _ = field_values(x_end)
    u_end_x = d_dx(u_end, x_end)
    loss_bc_end = torch.mean((c * u_end_x - P_load)**2)

    Loss_1 = loss_u_phy + loss_bc0
    Loss_2 = loss_phi_phy + loss_bc0
    Loss_3 = loss_n_phy + loss_bc0
    Loss_4 = loss_theta_phy + loss_bc0

    # Eq. (4): perdida total = promedio de las M subtareas
    total = (Loss_1 + Loss_2 + Loss_3 + Loss_4) / 4 + loss_bc_end
    return total, dict(u=Loss_1.item(), phi=Loss_2.item(), n=Loss_3.item(), theta=Loss_4.item())

## 4. Entrenamiento conjunto de las 4 subredes

In [ ]:
params = list(net_u.parameters()) + list(net_phi.parameters()) + \
         list(net_n.parameters()) + list(net_theta.parameters())
optimizer = torch.optim.Adam(params, lr=1e-3)
history = []

for epoch in range(4000):
    optimizer.zero_grad()
    loss, parts = compute_losses()
    loss.backward()
    optimizer.step()
    history.append(loss.item())
    if epoch % 500 == 0:
        print(f'epoch {epoch:5d} | loss={loss.item():.4e} | subtasks={ {k: round(v,4) for k,v in parts.items()} }')

## 5. Resultados: los 4 campos acoplados a lo largo de la varilla

In [ ]:
x_plot = torch.linspace(0, a, 200, device=device).view(-1, 1)
with torch.no_grad():
    u_p, phi_p, n_p, theta_p = field_values(x_plot)
x_np = x_plot.cpu().numpy().flatten()

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, field, title in zip(axes,
                             [u_p, phi_p, n_p, theta_p],
                             ['Desplazamiento $u$ (FLS-1)', 'Potencial $\\varphi$ (FLS-2)',
                              'Portadores $n$ (FLS-3)', 'Temperatura $\\theta$ (FLS-4)']):
    ax.plot(x_np, field.cpu().numpy().flatten())
    ax.set_xlabel('$x$')
    ax.set_title(title)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
plt.semilogy(history)
plt.xlabel('Epoca')
plt.ylabel('Loss total (promedio de subtareas, Eq. 4)')
plt.title('Convergencia de DD-PINNs-TD (4 subredes acopladas)')
plt.grid(alpha=0.3)
plt.show()

Cada subred aprende su propio campo fisico usando una perdida propia (Eq. 1-3), pero las cuatro permanecen acopladas porque cada residuo de subtarea depende de las salidas de las otras subredes (p. ej. la perdida de $u$ depende de $\varphi_{,xx}$ y $\theta_{,x}$). Esta es la esencia del enfoque **DD-PINNs-TD**: descomponer un problema multi-fisica en subtareas manejables por redes pequen~as e independientes, en vez de una unica red monolitica que prediga todos los campos simultaneamente.